# Résoudre `5_energy` avec OR-Tools
Cinquième jeu de données : 11 000 modèles et 100 000 datasets. Le commentaire de l'instance annonce : *« Looks like there's a discount on our electricity bill »*. L'énergie n'est plus la contrainte. Ce sont les **données** qui manquent.

Ce notebook montre trois choses nouvelles :
1. les **sources** changent de rôle : elles servent maintenant à **économiser des données**, et le modèle doit les autoriser sur **tous** les besoins ;
2. une **astuce de modélisation** : les sources d'un même type sont interchangeables, on peut donc les agréger ;
3. un **piège numérique** : une contrainte inutile, avec d'énormes coefficients, fait annoncer au solveur un faux optimum.

**Prérequis** : `datasets/5_energy.json`, `test_solution.py`, `pandas`, `ortools`.

**Durée d'exécution** : environ 12 minutes (dont 7 pour le solveur).

## 1. Charger l'instance et le repère

In [1]:
from pathlib import Path
from collections import defaultdict, Counter
import sys, json, time, bisect, random
import pandas as pd
from ortools.linear_solver import pywraplp
pd.set_option("display.width", 150)

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))
from test_solution import getSolutionScore

DATASET = ROOT_DIR / "datasets" / "5_energy.json"
DATASET_TXT = DATASET.read_text()
instance = json.loads(DATASET_TXT)
energy_cap = instance["energyCap"]
print("commentaire de l'instance :", instance["comment"])

VALUE  = {m["id"]: m["value"] for m in instance["models"]}
ENERGY = {m["id"]: m["energyCost"] for m in instance["models"]}
NEEDS  = {m["id"]: [(t, b["lowerBound"], b["upperBound"]) for t, b in m["dataRequirements"].items()] for m in instance["models"]}
SIZE   = {d["id"]: d["size"] for d in instance["datasets"]}
TYPE   = {d["id"]: d["dataType"] for d in instance["datasets"]}
COPY   = {d["id"]: d["isCopyrighted"] for d in instance["datasets"]}
IMPOSSIBLE = {m for m, needs in NEEDS.items() if any(lo > hi for _, lo, hi in needs)}
LOWER = {m: {t: lo for t, lo, hi in needs if lo <= hi} for m, needs in NEEDS.items()}
MONO = {m: needs[0][0] for m, needs in NEEDS.items() if len(needs) == 1 and m not in IMPOSSIBLE}
FREE_SUPPLY  = {t: sum(SIZE[d] for d in SIZE if TYPE[d] == t and COPY[d] == 0) for t in "ntic"}
TOTAL_SUPPLY = {t: sum(SIZE[d] for d in SIZE if TYPE[d] == t) for t in "ntic"}

WINNER = 3_786_108
print(f"{len(VALUE)} modèles, {len(SIZE)} datasets | gagnant du concours : {WINNER:,}")

commentaire de l'instance : Looks like there's a discount on our electricity bill.
11000 modèles, 100000 datasets | gagnant du concours : 3,786,108


## 2. Ce qui change

In [2]:
print(f"énergie de TOUS les modèles : {sum(ENERGY.values()) / energy_cap:.2f} × le plafond")
print(f"datasets sous droits : {sum(COPY.values()) / len(COPY):.0%}")
print(f"modèles à un seul besoin (sources possibles) : {len(MONO)} | modèles impossibles : {len(IMPOSSIBLE)}")
demand = {t: sum(LOWER[m].get(t, 0) for m in VALUE) for t in "ntic"}
pd.DataFrame({"offre_totale": TOTAL_SUPPLY, "offre_libre": FREE_SUPPLY, "demande_si_tout": demand,
              "demande_sur_offre": {t: round(demand[t] / TOTAL_SUPPLY[t], 2) for t in "ntic"},
              "demande_sur_offre_libre": {t: round(demand[t] / FREE_SUPPLY[t], 2) for t in "ntic"}})

énergie de TOUS les modèles : 0.92 × le plafond
datasets sous droits : 66%
modèles à un seul besoin (sources possibles) : 4554 | modèles impossibles : 231


,offre_totale,offre_libre,demande_si_tout,demande_sur_offre,demande_sur_offre_libre
n,126157240,42677701,127858667,1.01,3.00
t,124566959,43060085,129409873,1.04,3.01
i,124156184,43086699,134235905,1.08,3.12
c,124375133,41707631,129601799,1.04,3.11


**Lecture**
- L'énergie de **tous** les modèles réunis tient sous le plafond (0,92). On peut entraîner n'importe quel ensemble de modèles : **l'énergie ne compte plus**.
- La demande dépasse l'offre totale de données (environ 1,05 fois) et vaut 3 fois l'offre libre. On ne pourra pas servir tout le monde, et moins de la moitié des modèles pourra être à pleine valeur.
- 4 554 modèles n'ont qu'un seul besoin remplissable : ce sont autant de sources possibles.

Le problème devient : **avec des données limitées, quels modèles servir, à quelle valeur, et avec quelles sources ?**

## 3. Les sources changent de rôle
Jusqu'ici, une source servait à nourrir un modèle **impossible**. Ici, elle a un autre intérêt. Une source de type $t$ couvre un besoin de type $t$ **sans consommer de datasets** pour ce besoin. La source a elle-même besoin de données, mais son besoin peut être bien plus petit que celui qu'elle couvre. Quand les données manquent, c'est une **économie de données**.

Le prix : la cible passe à demi-valeur, et la valeur de la source est perdue. Mais sur cette instance, plus de la moitié des modèles seront à demi-valeur de toute façon, faute de données libres. Pour eux, être une cible ne coûte rien.

Dans les notebooks 11 à 13, le modèle n'autorisait les sources que pour les besoins impossibles. Est-ce que cela suffit ici ?

## 4. Le modèle, avec des sources agrégées
Avec 4 554 sources possibles et 22 000 besoins, des variables $u_{s,m,t}$ « la source $s$ couvre le besoin $t$ de $m$ » se compteraient en **millions**. Mais une source de type $t$ peut couvrir **n'importe quel** besoin de type $t$ : seul le type compte. On **agrège** donc :
- $w_s$ : le modèle $s$ sert de source (à quelle cible, on le décidera après) ;
- $c_{i,t}$ : le besoin $t$ du modèle $i$ est couvert par une source (seulement si $i$ est à demi-valeur : $c_{i,t} \le y_i$). Pour un besoin impossible, $c_{i,t} = y_i$ ;
- par type : $\sum_i c_{i,t} = \sum_{s \text{ de type } t} w_s$ (autant de besoins couverts que de sources).

Les données totales deviennent : $\sum L_{i,t}\,x_i + \sum L_{i,t}\,(y_i - c_{i,t}) + \sum L_{s,t}\,w_s \le \text{offre}_t$. On passe de millions de variables à environ 50 000.

Le paramètre `possible_sources` permet de comparer avec l'ancien modèle (sources pour les besoins impossibles seulement). Le paramètre `energy_constraint` sert à la partie 6.

In [3]:
def solve_model(seconds, possible_sources=True, energy_constraint=True):
    t0 = time.time()
    S = pywraplp.Solver.CreateSolver("SCIP")
    x = {m: S.BoolVar(f"x{m}") for m in VALUE if m not in IMPOSSIBLE}      # pleine valeur
    y = {m: S.BoolVar(f"y{m}") for m in VALUE}                              # demi-valeur
    w = {s: S.BoolVar(f"w{s}") for s in MONO}                               # source
    c = {}                                                                  # besoin couvert par une source
    for m, needs in NEEDS.items():
        for t, lo, hi in needs:
            if lo > hi:
                c[(m, t)] = y[m]
            elif possible_sources:
                c[(m, t)] = S.BoolVar(f"c{m}_{t}")
                S.Add(c[(m, t)] <= y[m])
    for m in VALUE:
        S.Add((x[m] if m in x else 0) + y[m] + (w[m] if m in w else 0) <= 1)   # un seul rôle
    if energy_constraint:
        S.Add(sum(ENERGY[m] * x[m] for m in x) + sum(ENERGY[m] * y[m] for m in y)
              + sum(ENERGY[s] * w[s] for s in w) <= energy_cap)
    for t in "ntic":
        S.Add(sum(var for (m, tt), var in c.items() if tt == t) == sum(w[s] for s in w if MONO[s] == t))
        S.Add(sum(LOWER[m][t] * x[m] for m in x if t in LOWER[m]) <= FREE_SUPPLY[t])
        S.Add(sum(LOWER[m][t] * x[m] for m in x if t in LOWER[m])
              + sum(LOWER[m][t] * (y[m] - c.get((m, t), 0)) for m in y if t in LOWER[m])
              + sum(LOWER[s][t] * w[s] for s in w if MONO[s] == t) <= TOTAL_SUPPLY[t])
    S.Maximize(sum(VALUE[m] * x[m] for m in x) + sum((VALUE[m] // 2) * y[m] for m in y))
    S.SetTimeLimit(seconds * 1000)
    params = pywraplp.MPSolverParameters()
    params.SetDoubleParam(params.RELATIVE_MIP_GAP, 0.0)
    status = S.Solve(params)
    val = lambda v: round(v.solution_value())
    sol = dict(full=[m for m in x if val(x[m])], half=[m for m in y if val(y[m])],
               sources=[s for s in w if val(w[s])], covered=[(m, t) for (m, t), var in c.items() if val(var)])
    obj, bound = S.Objective().Value(), S.Objective().BestBound()
    info = dict(statut="OPTIMAL" if status == pywraplp.Solver.OPTIMAL else "FEASIBLE", objectif=round(obj),
                borne=round(bound, 1), écart_prouvé=round(bound - obj, 1), secondes=round(time.time() - t0),
                pleins=len(sol["full"]), demi=len(sol["half"]), sources=len(sol["sources"]))
    return sol, info

solver_runs = {}

### Sources pour les besoins impossibles seulement

In [4]:
sol_old, solver_runs["sources : besoins impossibles seulement"] = solve_model(120, possible_sources=False)
pd.DataFrame(solver_runs).T

,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
sources : besoins impossibles seulement,OPTIMAL,3709518,3709518.0,0.0,19,4688,5221,47


**Lecture** : SCIP prouve l'optimum de ce modèle en quelques secondes, et il est **environ 77 000 points sous le gagnant**. Aucune solution de ce modèle ne peut rattraper le gagnant. **Le gagnant a donc forcément utilisé des sources sur des besoins remplissables.** C'est la borne qui le prouve : sans elle, on aurait pu croire à un problème de réglage.

## 5. Sources pour tous les besoins
On lance le modèle complet, avec la contrainte d'énergie, comme dans les notebooks précédents.

In [5]:
sol_energy, solver_runs["sources partout, avec contrainte d'énergie"] = solve_model(150, energy_constraint=True)
pd.DataFrame(solver_runs).T

,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
sources : besoins impossibles seulement,OPTIMAL,3709518,3709518.0,0.0,19,4688,5221,47
"sources partout, avec contrainte d'énergie",OPTIMAL,3780183,3780183.0,0.0,119,4645,5887,234


**Lecture** : le gain est énorme, de l'ordre de 70 000 points. Mais regardez bien le résultat : chez moi, SCIP annonce **OPTIMAL** à 3 780 183, soit **moins que le gagnant**. Si c'était vrai, le gagnant aurait fait mieux que l'optimum d'un modèle qui est une relaxation du vrai problème. C'est impossible.

(Selon la machine et la durée, SCIP peut aussi s'arrêter avant de conclure. Dans ce cas, comparez son objectif et sa borne au score du gagnant.)

## 6. Le piège numérique
D'où vient ce faux optimum ? Regardons la contrainte d'énergie. L'énergie de **tous** les modèles réunis tient sous le plafond : la contrainte ne peut **jamais** être violée. Elle est **redondante**. Mais elle contient 15 000 coefficients jusqu'à $10^{12}$, pour un second membre de $6 \times 10^{15}$.

Un solveur MIP calcule en nombres à virgule flottante, avec des tolérances (de l'ordre de $10^{-6}$ à $10^{-9}$). Avec de tels ordres de grandeur, les erreurs d'arrondi dans les coupes et les bornes peuvent devenir assez grandes pour **éliminer à tort** de bonnes solutions. Le solveur conclut alors à l'optimalité sur un problème légèrement faux.

Remède : **supprimer la contrainte redondante**.

In [6]:
print(f"énergie de tous les modèles / plafond : {sum(ENERGY.values()) / energy_cap:.3f} -> contrainte toujours satisfaite")
sol, solver_runs["sources partout, SANS contrainte d'énergie"] = solve_model(300, energy_constraint=False)
pd.DataFrame(solver_runs).T

énergie de tous les modèles / plafond : 0.920 -> contrainte toujours satisfaite


,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
sources : besoins impossibles seulement,OPTIMAL,3709518,3709518.0,0.0,19,4688,5221,47
"sources partout, avec contrainte d'énergie",OPTIMAL,3780183,3780183.0,0.0,119,4645,5887,234
"sources partout, SANS contrainte d'énergie",FEASIBLE,3786476,3786539.4,63.4,301,4638,5902,238


**Lecture**
- Sans la contrainte redondante, SCIP trouve une solution **au-dessus du gagnant** sur le papier, avec une borne à peine plus haute.
- Le modèle était juste ; c'est la **résolution** qui était fausse. Deux leçons pour un physicien habitué aux calculs numériques : **un solveur peut se tromper**, et **les ordres de grandeur comptent**. Mettre à l'échelle les données (par exemple exprimer l'énergie en milliards) ou retirer les contraintes inutiles fait partie du travail de modélisation.
- Le score du gagnant a encore servi de **témoin** : c'est lui qui a révélé l'erreur.

## 7. Attribuer les datasets
On reprend `Pool`, `fill` et la file d'attente unique des notebooks 12 et 13, avec deux nouveautés :
- les modèles à demi-valeur ne reçoivent des datasets que pour leurs besoins **non couverts** par une source ;
- après l'attribution, on **apparie** les sources et les besoins couverts, type par type. Si une cible n'a pas toutes ses sources, on la retire et on rend ses datasets.

In [7]:
class Pool:
    def __init__(self, ids):
        self.ids = sorted(ids, key=SIZE.get)
        self.sizes = [SIZE[d] for d in self.ids]
    def take(self, i):
        self.sizes.pop(i)
        return self.ids.pop(i)
    def put(self, d):
        i = bisect.bisect_left(self.sizes, SIZE[d])
        self.sizes.insert(i, SIZE[d]); self.ids.insert(i, d)
    def remove(self, d):
        i = bisect.bisect_left(self.sizes, SIZE[d])
        while self.ids[i] != d:
            i += 1
        self.take(i)

def find_pair(sizes, a, b):
    i, j, best = 0, bisect.bisect_right(sizes, b) - 1, None
    while i < j:
        s = sizes[i] + sizes[j]
        if s > b:
            j -= 1
        elif s < a:
            i += 1
        else:
            if best is None or s < best[0]:
                best = (s, i, j)
            j -= 1
    return best

def fill(pool, lower, upper):
    taken, total = [], 0
    while total < lower:
        a, b = lower - total, upper - total
        i = bisect.bisect_left(pool.sizes, a)
        if i < len(pool.sizes) and pool.sizes[i] <= b:
            total += pool.sizes[i]; taken.append(pool.take(i)); continue
        pair = find_pair(pool.sizes, a, b) if pool.sizes and a <= 2 * pool.sizes[-1] else None
        if pair:
            _, i, j = pair
            total += pool.sizes[i] + pool.sizes[j]
            taken.append(pool.take(j)); taken.append(pool.take(i)); continue
        j = bisect.bisect_right(pool.sizes, b) - 1
        if j < 0:
            break
        total += pool.sizes[j]; taken.append(pool.take(j))
    if lower <= total <= upper:
        return taken
    for d in taken:
        pool.put(d)
    return None

def give(pools, model, needs, statuses):
    got = []
    for t, lo, hi in needs:
        res = None
        for s in statuses:
            res = fill(pools[(t, s)], lo, hi)
            if res is not None:
                break
        if res is None:
            for d in got:
                pools[(TYPE[d], COPY[d])].put(d)
            return None
        got += res
    return [[int(d), int(model)] for d in got]

def pack(sol, order_key):
    """order_key(modèle, besoins à remplir) -> clé de tri. Rend (plan, nombre de modèles ratés)."""
    pools = {(t, c): Pool([d for d in SIZE if TYPE[d] == t and COPY[d] == c]) for t in "ntic" for c in (0, 1)}
    covered = defaultdict(set)
    for m, t in sol["covered"]:
        covered[m].add(t)
    jobs = [(m, NEEDS[m], (0,), "plein") for m in sol["full"]]
    jobs += [(m, [n for n in NEEDS[m] if n[0] not in covered[m]], (1, 0), "demi") for m in sol["half"]]
    jobs += [(s, NEEDS[s], (1, 0), "source") for s in sol["sources"]]
    data, failed = {}, []
    for m, needs, statuses, kind in sorted(jobs, key=lambda j: order_key(j[0], j[1])):
        got = give(pools, m, needs, statuses)
        if got is None:
            failed.append((m, kind))
        else:
            data[m] = got
    for m, kind in failed:                                   # seconde chance à demi-valeur pour les pleins
        if kind == "plein":
            got = give(pools, m, NEEDS[m], (1, 0))
            if got is not None:
                data[m] = got
    # Appariement sources -> besoins couverts, type par type
    available = defaultdict(list)
    for s in sol["sources"]:
        if s in data:
            available[MONO[s]].append(s)
    model_mappings = []
    for m in sol["half"]:
        if m not in data:
            continue
        pairs = []
        for t in covered[m]:
            if available[t]:
                pairs.append((available[t].pop(), m))
        if len(pairs) == len(covered[m]):
            model_mappings += pairs
        else:                                                # cible incomplète : on la retire
            for s, _ in pairs:
                available[MONO[s]].append(s)
            data.pop(m)
    for t, unused in available.items():                      # sources sans cible : on les retire
        for s in unused:
            data.pop(s)
    plan = {"dataMappings": [p for got in data.values() for p in got],
            "modelMappings": [[int(s), int(m)] for s, m in model_mappings]}
    return plan, len(failed)

### Plusieurs ordres d'attribution
Comme dans le notebook 13 : les deux ordres de référence, plus quelques ordres bruités. Ici, chaque attribution prend une vingtaine de secondes (100 000 datasets), on se limite donc à 4 ordres bruités.

In [8]:
orders = {"petits besoins": lambda m, needs: sum(lo for t, lo, hi in needs),
          "plus contraints": lambda m, needs: (min((hi for t, lo, hi in needs), default=0), sum(lo for t, lo, hi in needs))}
rng = random.Random(0)
for k in range(4):
    noise = {m: rng.uniform(0.8, 1.2) for m in VALUE}
    orders[f"bruité {k}"] = lambda m, needs, nz=noise: sum(lo for t, lo, hi in needs) * nz[m]

rows, best = [], (0, None, None)
t0 = time.time()
for name, key in orders.items():
    plan, n_failed = pack(sol, key)
    score = getSolutionScore(json.dumps(plan), DATASET_TXT, show_details=False)[0]
    rows.append(dict(ordre=name, ratés=n_failed, score=score, écart_au_gagnant=score - WINNER))
    if score > best[0]:
        best = (score, name, plan)
print(f"{len(orders)} ordres en {time.time() - t0:.0f} s")
pd.DataFrame(rows).set_index("ordre")

6 ordres en 128 s


,ratés,score,écart_au_gagnant
ordre,,,
petits besoins,9,3785223,-885
plus contraints,5,3782084,-4024
bruité 0,9,3784246,-1862
bruité 1,10,3784242,-1866
bruité 2,9,3784747,-1361
bruité 3,10,3784721,-1387


## 8. Compléter et réparer
Deux étapes, avec la classe `State` du notebook 11 (sources et cibles gelées) :
1. **Compléter** : l'énergie ne limite pas, donc les datasets restants peuvent accueillir des modèles **non choisis**. On les essaie par valeur par unité de donnée décroissante, à pleine valeur si possible, sinon à demi-valeur ;
2. **Réparer** : comme dans les notebooks 11 et 13. Ici, il y a environ 5 900 modèles à demi-valeur, et tester tous les partenaires prendrait des heures. On revient à une limite, fixée à **20 partenaires**.

In [9]:
class State:
    """Solution modifiable, avec journal d'annulation. Sources et cibles sont gelées."""
    def __init__(self, plan):
        frozen_models = {x for pair in plan["modelMappings"] for x in pair}
        self.frozen_data = [p for p in plan["dataMappings"] if p[1] in frozen_models]
        self.frozen_sources = plan["modelMappings"]
        self.frozen = frozen_models
        self.data = {}
        for d, m in plan["dataMappings"]:
            if m not in frozen_models:
                self.data.setdefault(m, []).append(d)
        used = {d for d, _ in plan["dataMappings"]}
        self.pools = {(t, c): Pool([d for d in SIZE if d not in used and TYPE[d] == t and COPY[d] == c])
                      for t in "ntic" for c in (0, 1)}
        self.full = {m: not any(COPY[d] for d in ds) for m, ds in self.data.items()}
        self.log = []

    def remove(self, m):
        ds, full = self.data.pop(m), self.full.pop(m)
        for d in ds:
            self.pools[(TYPE[d], COPY[d])].put(d)
        self.log.append(("remove", m, ds, full))

    def add(self, m, statuses):                      # pas de test d'énergie : elle ne limite pas ici
        got = give(self.pools, m, NEEDS[m], statuses)
        if got is None:
            return False
        self.data[m] = [d for d, _ in got]
        self.full[m] = not any(COPY[d] for d in self.data[m])
        self.log.append(("add", m, None, None))
        return True

    def undo(self):
        while self.log:
            op, m, ds, full = self.log.pop()
            if op == "add":
                for d in self.data.pop(m):
                    self.pools[(TYPE[d], COPY[d])].put(d)
                self.full.pop(m)
            else:
                for d in ds:
                    self.pools[(TYPE[d], COPY[d])].remove(d)
                self.data[m], self.full[m] = ds, full

    def plan(self):
        mappings = [[int(d), int(m)] for m, ds in self.data.items() for d in ds] + self.frozen_data
        return {"dataMappings": mappings, "modelMappings": self.frozen_sources}

TYPES_OF = {m: {t for t, _, _ in needs} for m, needs in NEEDS.items()}
gain_full = lambda m: VALUE[m] - VALUE[m] // 2

def complete(s):
    candidates = [m for m in VALUE if m not in s.data and m not in s.frozen and m not in IMPOSSIBLE]
    candidates.sort(key=lambda m: -VALUE[m] / max(1, sum(LOWER[m].values())))
    added = 0
    for m in candidates:
        s.log = []
        if s.add(m, (0,)) or s.add(m, (1, 0)):
            added += 1
    return added

def repair(s, max_partners=20):
    n_repairs, improved = 0, True
    while improved:
        improved = False
        for b in sorted([m for m in s.data if not s.full[m]], key=gain_full, reverse=True):
            if b not in s.data or s.full[b]:
                continue
            s.log = []; s.remove(b)
            if s.add(b, (0,)):
                n_repairs += 1; improved = True; continue
            s.undo()
            partners = sorted([a for a in s.data if s.full[a] and TYPES_OF[a] & TYPES_OF[b] and gain_full(a) < gain_full(b)],
                              key=gain_full)
            for a in partners[:max_partners]:
                s.log = []; s.remove(a); s.remove(b)
                if s.add(b, (0,)) and s.add(a, (1, 0)):
                    n_repairs += 1; improved = True; break
                s.undo()
    return n_repairs

best_score, best_name, best_plan = best
t0 = time.time()
state = State(best_plan)
added = complete(state)
score_completed = getSolutionScore(json.dumps(state.plan()), DATASET_TXT, show_details=False)[0]
n = repair(state)
final_plan = state.plan()
final_score = getSolutionScore(json.dumps(final_plan), DATASET_TXT, show_details=False)[0]
print(f"ordre retenu : {best_name} | {best_score:,}")
print(f"complétion : +{added} modèles -> {score_completed:,}")
print(f"réparation : {n} échanges -> {final_score:,}   ({time.time() - t0:.0f} s)")

ordre retenu : petits besoins | 3,785,223
complétion : +5 modèles -> 3,785,228
réparation : 8 échanges -> 3,785,420   (59 s)


## 9. Comparaison et vérification

In [10]:
info = solver_runs["sources partout, SANS contrainte d'énergie"]
summary = pd.DataFrame({"score": [final_score, info["objectif"], WINNER, info["borne"],
                                  solver_runs["sources : besoins impossibles seulement"]["objectif"]]},
                       index=["plan final", "sélection du solveur (avant datasets)", "gagnant du concours",
                              "borne du solveur", "optimum sans sources sur besoins remplissables"])
summary["écart_au_gagnant"] = summary.score - WINNER
summary["part_du_gagnant"] = (summary.score / WINNER).map("{:.3%}".format)
display(summary)

score, valid, message = getSolutionScore(json.dumps(final_plan), DATASET_TXT, show_details=True)
print("score officiel =", score, "| valide =", valid, message)

,score,écart_au_gagnant,part_du_gagnant
plan final,3785420.0,-688.0,99.982%
sélection du solveur (avant datasets),3786476.0,368.0,100.010%
gagnant du concours,3786108.0,0.0,100.000%
borne du solveur,3786539.4,431.4,100.011%
optimum sans sources sur besoins remplissables,3709518.0,-76590.0,97.977%



Total value: 3785420
Energy used: 5542149383490122 / 6048626138154986 (91.63%)
Number of used models: 10781 / 11000 (98.01%)
  Number of devalued models: 6145 (57.00%)
  Number of models used as data source: 238 (2.21%)
Number of used datasets: 99129 / 100000 (99.13%)
score officiel = 3785420 | valide = True 


**Lecture**
- On arrive à **99,98 %** du score du gagnant, à quelques centaines de points.
- La sélection du solveur dépasse le gagnant sur le papier : tout l'écart vient de l'**attribution des datasets**. Les données libres sont utilisées à près de 100 %, et une dizaine de modèles pleins ne trouvent pas leurs données libres exactes.
- Pistes pour aller plus loin : la boucle du notebook 11 (interdire la pleine valeur aux modèles qui ne se casent pas, puis relancer le solveur), ou une petite **marge** sur les données libres dans le modèle pour absorber les pertes de découpage.

## 10. Écrire le fichier de solution

In [11]:
out_dir = ROOT_DIR / "solutions"; out_dir.mkdir(exist_ok=True)
out = out_dir / "5_energy_main.json"
if valid:
    out.write_text(json.dumps(final_plan))
    print("écrit :", out)

écrit : /home/claude/proj/solutions/5_energy_main.json


## À retenir
1. **Une règle prend un sens différent selon les données.** Les sources nourrissaient des modèles impossibles ; ici, elles **économisent des données**. Le modèle doit les autoriser partout.
2. **La borne a prouvé ce qui manquait.** L'ancien modèle avait un optimum prouvé 77 000 points sous le gagnant : il ne pouvait pas être juste.
3. **Agréger des objets interchangeables.** Les sources d'un même type sont équivalentes. Au lieu de millions de variables « qui sert qui », on compte les sources et les besoins couverts par type, et on apparie après.
4. **Un solveur peut se tromper.** Une contrainte inutile avec des coefficients de $10^{12}$ a produit un faux « OPTIMAL ». Retirer les contraintes redondantes et mettre les données à l'échelle fait partie de la modélisation.
5. **Toujours confronter le résultat à un témoin.** Ici, le score du gagnant a servi de témoin. Sans lui, on aurait cru à l'optimum.